# **Предобработка данных для анализа заболеваний сердца**

## **1.	Очистка данных**

### **1.1.	Обработка пропущенных значений**

* __Цель:__ Обеспечить целостность набора данных, исключив влияние пустых значений на статистические показатели и алгоритмы машинного обучения.
* __Задачи:__
  - Обнаружить и классифицировать пропуски.
  - Принять решение о стратегии их заполнения или удаления.
* __Алгоритм использования:__
  1. __Анализ:__ Определить количество и процент пропусков в каждом столбце.
  2. __Диагностика:__ Определить природу пропусков (случайные/системные).
  3. __Применение стратегии:__
     * __Удаление:__ Исключение из набора данных строк или столбцов (признаков), в которых доля пропусков превышает критический порог.
     * __Заполнение:__
       * __Константой__ (например, "Не указано", 0).
       * __Статистическими мерами:__
         * Числовые данные: Заполнение средним значением или медианой;
         * Категориальные данные: Заполнение модой (наиболее вероятное значение).
       * __Значением из следующей или предыдущей строки__ для временных рядов.

In [ ]:
# --- 1. ЗАГРУЗКА И ПЕРВИЧНАЯ НАСТРОЙКА ---

# Установка и загрузка необходимых библиотек
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown

# Конфигурация логирования
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
# Загрузка набора данных
try:
    data = pd.read_csv("../data/raw/heart_disease_uci.csv")
    display(Markdown("#### **Содержание набора данных:**"))
    display(data.head())
    display(Markdown("#### **Структура набора данных:**"))
    data.info()
except FileNotFoundError:
    logging.error(
        "Файл '../data/raw/heart_disease_uci.csv' не найден. Убедитесь, что скрипт загрузки данных был запущен."
    )

In [ ]:
# --- 2. ФУНКЦИИ ДЛЯ АНАЛИЗА И ОБРАБОТКИ ---


def get_missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Создает сводную таблицу по пропущенным значениям."""
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_summary = pd.DataFrame(
        {"Missing Values": missing_count, "Percentage (%)": missing_percent}
    )
    missing_summary = missing_summary.sort_values(by="Percentage (%)", ascending=False)
    return missing_summary

In [ ]:
# --- 3. КОНФИГУРАЦИЯ И ОПРЕДЕЛЕНИЕ ТИПОВ ПРИЗНАКОВ ---

# Группировка признаков по типу
ALL_COLUMNS = data.columns
FEATURE_CONFIG = {
    "ordinal": ["restecg", "slope", "ca"],  # Категориальные признаки с порядком
    "binary": [
        "sex",
        "fbs",
        "exang",
    ],  # Категориальные признаки с двумя категориями (бинарные)
    "nominal": [
        "dataset",
        "cp",
        "thal",
    ],  # Категориальные признаки без порядка (номинальные)
    "numerical": [
        "age",
        "trestbps",
        "chol",
        "thalch",
        "oldpeak",
    ],  # Количественные признаки (числовые)
}

for group, columns in FEATURE_CONFIG.items():
    FEATURE_CONFIG[group] = [col for col in columns if col in ALL_COLUMNS]

In [ ]:
# --- 4. АНАЛИЗ ПРОПУСКОВ "ДО" ---

display(Markdown("#### **Анализ пропущенных значений ДО обработки**"))
missing_summary_before = get_missing_summary(data)
display(missing_summary_before)

In [ ]:
# Визуализация пропусков с помощью тепловой карты (heatmap)
plt.figure(figsize=(16, 6))
sns.heatmap(data.isnull(), cbar=False, yticklabels=False, cmap="cubehelix_r")
plt.title("Тепловая карта пропусков (heatmap)", fontsize=14)
plt.show()

In [ ]:
# --- 5. ПРИМЕНЕНИЕ СТРАТЕГИИ ---

display(Markdown("#### **Применение стратегии обработки пропущенных значений**"))

# Создание копии набора данных для дальнейшей обработки пропущенных значений
data_cleaned = data.copy()
THRESHOLD = 50  # Установка порога (50%)

# 1.  Удаление столбцов, в которых процент пропусков превышает установленный порог в 50%
cols_to_drop = missing_summary_before[
    missing_summary_before["Percentage (%)"] > THRESHOLD
].index

if not cols_to_drop.empty:
    data_cleaned = data_cleaned.drop(columns=cols_to_drop)
    display(
        Markdown(
            f"* **Признаки** (`{', '.join(cols_to_drop)}`) были удалены из-за высокого процента пропущенных значений **(более {THRESHOLD}%)**."
        )
    )
else:
    display(
        Markdown(
            "* Признаков, где процент пропущенных значений превышает установленный порог не найдено."
        )
    )

# Обновление списков признаков после удаления столбцов с высоким процентом пропусков
numerical_cols = [
    col for col in FEATURE_CONFIG["numerical"] if col in data_cleaned.columns
]
categorical_cols = [
    col
    for group in ["ordinal", "binary", "nominal"]
    for col in FEATURE_CONFIG[group]
    if col in data_cleaned.columns
]

# 2. Заполнение пропусков для числовых признаков медианными значениями (медианой)
if numerical_cols:
    data_cleaned[numerical_cols] = data_cleaned[numerical_cols].fillna(
        data_cleaned[numerical_cols].median()
    )
    display(
        Markdown(
            f"* **Числовые признаки** (`{', '.join(numerical_cols)}`) заполнены **медианными значениями (медианой)**."
        )
    )

# 3. Заполнение пропусков для категориальных признаков наиболее частыми значениями (модой)
if categorical_cols:
    for col in categorical_cols:
        data_cleaned[col] = data_cleaned[col].fillna(data_cleaned[col].mode()[0])
    display(
        Markdown(
            f"* **Категориальные признаки** (`{', '.join(categorical_cols)}`) заполнены **наиболее частыми значениями (модой)**."
        )
    )

In [ ]:
# --- 6. АНАЛИЗ ПРОПУСКОВ "ПОСЛЕ" ---

display(Markdown("#### **Анализ пропущенных значений ПОСЛЕ обработки**"))
missing_summary_after = get_missing_summary(data_cleaned)
if missing_summary_after["Missing Values"].sum() == 0:
    display(Markdown("Пропущенные значения в данных отсутствуют."))
else:
    display(Markdown("После обработки в данных все еще есть пропущенные значения:"))
    display(missing_summary_after[missing_summary_after["Missing Values"] > 0])

### **1.2.	Обработка дубликатов**

* __Цель:__ Исключить искажение результатов анализа за счет повторяющихся записей.
* __Задачи:__ Найти и удалить полностью дублирующиеся строки или строки, дублирующиеся по ключевым полям.
* __Алгоритм использования:__
  1. __Поиск:__ Выявление дубликатов во всем наборе данных или по заданным столбцам.
  2. __Анализ:__ Просмотр найденных дубликатов для принятия решения.
  3. __Удаление:__ Удаление всех повторяющихся записей, кроме первого или последнего вхождения.

In [ ]:
# --- 1. ПОИСК И АНАЛИЗ ДУБЛИКАТОВ ---

# Формирование списка столбцов для проверки на дублирующиеся строки (исключая 'id')
cols_to_check = [col for col in data_cleaned.columns if col != "id"]

# Маска для всех копий дубликатов
duplicate_mask = data_cleaned.duplicated(subset=cols_to_check, keep=False)

# Подсчет количества дубликатов, которые подлежат удалению
num_duplicates_to_drop = data_cleaned.duplicated(subset=cols_to_check).sum()

# Просмотр найденных дубликатов для принятия решения
if num_duplicates_to_drop > 0:
    display(Markdown("#### **Найденные дублирующиеся строки (все копии):**"))
    display(data_cleaned[duplicate_mask].sort_values(by=cols_to_check))
else:
    display(Markdown("**Дублирующиеся записи не найдены.**"))

In [ ]:
# --- 2. ПРИМЕНЕНИЕ СТРАТЕГИИ: УДАЛЕНИЕ ---

if num_duplicates_to_drop > 0:
    # Сохранение размерности данных ДО удаления
    shape_before = data_cleaned.shape

    # Удаление дубликатов и сброс системного индекса Pandas
    data_cleaned = data_cleaned.drop_duplicates(subset=cols_to_check).reset_index(
        drop=True
    )

    # Проверка на наличие дубликатов после удаления
    num_duplicates_after = data_cleaned.duplicated(subset=cols_to_check).sum()

    display(Markdown(f"Найдено дублирующихся записей: **{num_duplicates_to_drop}**."))
    display(
        Markdown(f"Количество дубликатов после удаления: **{num_duplicates_after}**.")
    )
    display(
        Markdown(
            f"Размер набора данных изменился с **{shape_before[0]}** на **{data_cleaned.shape[0]}** строк."
        )
    )
    display(
        Markdown(
            f"Размер набора данных ПОСЛЕ удаления: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
        )
    )

### **1.3.	Обработка выбросов**

* __Цель:__ Уменьшить влияние аномальных значений, которые могут быть следствием ошибок ввода или редких событий, на статистические характеристики и модели.
* __Задачи:__ Определить аномалии и скорректировать их.
* __Алгоритм использования:__
  1. __Обнаружение:__
     * __Визуальные методы:__ Ящик с усами (Boxplot), гистограммы.
     * __Статистические методы:__
       * __Метод межквартильного размаха:__ Выбросами считаются значения, выходящие за пределы [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR].
       * __Метод Z-отклонения:__ Выбросами считаются значения, чей модуль Z-оценки больше 3 (при нормальном распределении).
  2. __Обработка:__
     *  Удаление (если выбросы — результат ошибки).
     *  Замена на граничные значения.
     *  Замена на среднее или медиану.

In [ ]:
# --- 1. ФУНКЦИИ ДЛЯ АНАЛИЗА И ОБРАБОТКИ ВЫБРОСОВ ---


def get_outlier_summary(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Рассчитывает границы по методу IQR, количество и процент выбросов.
    """
    # Вычисление Q1, Q3 и IQR для каждого числового признака
    Q1 = df[columns].quantile(0.25)
    Q3 = df[columns].quantile(0.75)
    IQR = Q3 - Q1

    # Определение границ для выбросов
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Подсчет количества выбросов по каждому числовому признаку
    outliers = (df[columns] < lower_bound) | (df[columns] > upper_bound)
    outlier_counts = outliers.sum()
    outlier_percentage = (outlier_counts / len(df)) * 100

    summary_df = pd.DataFrame(
        {
            "Q1": Q1,
            "Q3": Q3,
            "IQR": IQR,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": outlier_counts,
            "Outlier Percentage (%)": outlier_percentage,
        }
    )
    return summary_df


def plot_outliers(df: pd.DataFrame, columns: list, title_prefix: str):
    """Строит Boxplot для каждого числового признака."""
    plt.figure(figsize=(16, 6))
    sns.boxplot(data=df[columns], palette="Set2")
    plt.title(title_prefix, fontsize=14)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# --- 2. АНАЛИЗ ВЫБРОСОВ ДО ОБРАБОТКИ ---

# Визуализация выбросов с помощью boxplot для каждого числового признака ДО обработки выбросов
plot_outliers(
    data_cleaned,
    numerical_cols,
    "Ящик с усами (Boxplot) для каждого числового признака ДО обработки выбросов",
)

# Вывод статистики по выбросам по каждому числовому признаку
outlier_summary_before = get_outlier_summary(data_cleaned, numerical_cols)
display(Markdown("#### **Статистика по выбросам ДО обработки:**"))
display(outlier_summary_before)
display(
    Markdown(
        f"Размер данных ДО обработки выбросов: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
    )
)

In [ ]:
# --- 3. ПРИМЕНЕНИЕ СТРАТЕГИИ ОБРАБОТКИ ---
display(Markdown("#### **Применение стратегии обработки выбросов**"))

# Определение порога для обработки выбросов (5% от общего количества данных)
THRESHOLD = 5.0

for col in numerical_cols:
    summary = outlier_summary_before.loc[col]
    if summary["Outlier Count"] == 0:
        continue

    lower = summary["Lower Bound"]
    upper = summary["Upper Bound"]
    percentage = summary["Outlier Percentage (%)"]

    if percentage > THRESHOLD:
        # Замена выбросов на граничные значения для признаков с большим количеством выбросов (>5%)
        data_cleaned[col] = data_cleaned[col].clip(lower=lower, upper=upper)
        display(
            Markdown(
                f"* **{col}:** процент выбросов **более 5%** от общего количества данных ({percentage:.2f}%). Выбросы заменены на **граничные значения**."
            )
        )
    else:
        # Векторизованная замена выбросов на медианные значения для признаков с меньшим количеством выбросов (<=5%)
        median_value = data_cleaned[col].median()
        data_cleaned[col] = np.where(
            (data_cleaned[col] < lower) | (data_cleaned[col] > upper),
            median_value,
            data_cleaned[col],
        )
        display(
            Markdown(
                f"* **{col}:** процент выбросов **менее 5%** от общего количества данных ({percentage:.2f}%). Выбросы заменены на **медианные значения**."
            )
        )

In [ ]:
# --- 4. АНАЛИЗ ВЫБРОСОВ ПОСЛЕ ОБРАБОТКИ ---

# Визуализация выбросов с помощью boxplot для каждого числового признака ПОСЛЕ обработки выбросов
plot_outliers(
    data_cleaned,
    numerical_cols,
    "Ящик с усами (Boxplot) для каждого числового признака ПОСЛЕ обработки выбросов",
)

# Вывод статистики по выбросам по каждому числовому признаку
outlier_summary_after = get_outlier_summary(data_cleaned, numerical_cols)
display(Markdown("#### **Статистика по выбросам ПОСЛЕ обработки:**"))
display(outlier_summary_after)
display(
    Markdown(
        f"Размер данных ПОСЛЕ обработки выбросов: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
    )
)

In [ ]:
from pathlib import Path

PROCESSED_DATA_PATH = Path("../data/interim")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Сохранение очищенного набора данных в новый CSV файл
file_path = PROCESSED_DATA_PATH / "heart_disease_uci_cleaned.csv"
data_cleaned.to_csv(file_path, index=False)

# Проверка сохранения данных
display(Markdown(f"Данные успешно сохранены в файл `{file_path}`"))
display(
    Markdown(
        f"Итоговый размер набора данных: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
    )
)